# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## NB_P3_04 — MCD-rPPG raw dataset exploration (Phase 3)

Confirms the on-disk layout of the HuggingFace MCD-rPPG download before writing
the preprocessing loader: the directory structure, the file-type distribution
(to locate videos, PPG/ECG signals, and biomarker tables), and how recordings
are organized across subjects, states, and the three camera views.

In [1]:
from pathlib import Path
from collections import Counter

MCD_DIR = Path('G:/rppg/MCD_rPPG_dataset')
print('MCD dir exists:', MCD_DIR.exists())

print('\nTop-level entries:')
for entry in sorted(MCD_DIR.iterdir())[:30]:
    kind = 'dir ' if entry.is_dir() else 'file'
    size = '' if entry.is_dir() else f'  ({entry.stat().st_size / 1e6:.1f} MB)'
    print(f'  [{kind}] {entry.name}{size}')

ext = Counter()
total = 0
for p in MCD_DIR.rglob('*'):
    if p.is_file():
        ext[p.suffix.lower()] += 1
        total += 1
print(f'\nTotal files: {total}')
print('By extension:')
for e, n in ext.most_common(20):
    print(f'  {e or "(no ext)"}: {n}')

print('\nSample file paths (relative):')
shown = 0
for p in MCD_DIR.rglob('*'):
    if p.is_file():
        print('  ', p.relative_to(MCD_DIR))
        shown += 1
        if shown >= 30:
            break

MCD dir exists: True

Top-level entries:
  [dir ] .cache
  [file] .gitattributes  (0.3 MB)
  [file] db.csv  (0.9 MB)
  [dir ] ecg
  [dir ] meta
  [dir ] ppg
  [dir ] ppg_sync
  [file] README.md  (0.0 MB)
  [file] readme.txt  (0.0 MB)
  [dir ] video

Total files: 26411
By extension:
  .metadata: 13204
  .txt: 7201
  .avi: 3600
  .json: 1201
  .pw: 1200
  (no ext): 2
  .md: 1
  .csv: 1
  .tag: 1

Sample file paths (relative):
   README.md
   .gitattributes
   db.csv
   readme.txt
   ecg\1020_after.json
   ecg\1020_before.json
   ecg\1024_after.json
   ecg\1024_before.json
   ecg\1035_after.json
   ecg\1035_before.json
   ecg\1091_after.json
   ecg\1091_before.json
   ecg\1097_after.json
   ecg\1097_before.json
   ecg\1099_after.json
   ecg\1099_before.json
   ecg\1107_after.json
   ecg\1107_before.json
   ecg\1113_after.json
   ecg\1113_before.json
   ecg\1115_after.json
   ecg\1115_before.json
   ecg\1149_after.json
   ecg\1149_before.json
   ecg\1156_after.json
   ecg\1176_after.json
 

##  Per-recording linkage and file formats

Inspects how each recording's pieces connect and what format they are in: the
filename patterns in each subdirectory (to see how the three camera views and
the two states are named), the db.csv biomarker columns, and the on-disk format
of one ECG, PPG, ppg_sync, and meta file. This is what the loader will parse.

In [2]:
import pandas as pd

for sub in ['video', 'ppg', 'ppg_sync', 'meta', 'ecg']:
    d = MCD_DIR / sub
    if d.exists():
        names = sorted(p.name for p in d.iterdir() if p.is_file())
        print(f'{sub}/  ({len(names)} files)')
        for nm in names[:6]:
            print('   ', nm)
        print()

df_db = pd.read_csv(MCD_DIR / 'db.csv')
print('db.csv shape:', df_db.shape)
print('columns:', list(df_db.columns))
print(df_db.head(3).to_string())

video/  (3600 files)
    1020_FullHDwebcam_after.avi
    1020_FullHDwebcam_before.avi
    1020_IriunWebcam_after.avi
    1020_IriunWebcam_before.avi
    1020_USBVideo_after.avi
    1020_USBVideo_before.avi

ppg/  (1200 files)
    1020_after.PW
    1020_before.PW
    1024_after.PW
    1024_before.PW
    1035_after.PW
    1035_before.PW

ppg_sync/  (3600 files)
    1020_FullHDwebcam_after.txt
    1020_FullHDwebcam_before.txt
    1020_IriunWebcam_after.txt
    1020_IriunWebcam_before.txt
    1020_USBVideo_after.txt
    1020_USBVideo_before.txt

meta/  (3600 files)
    1020_FullHDwebcam_after.txt
    1020_FullHDwebcam_before.txt
    1020_IriunWebcam_after.txt
    1020_IriunWebcam_before.txt
    1020_USBVideo_after.txt
    1020_USBVideo_before.txt

ecg/  (1200 files)
    1020_after.json
    1020_before.json
    1024_after.json
    1024_before.json
    1035_after.json
    1035_before.json

db.csv shape: (3600, 25)
columns: ['patient_id', 'weight', 'height', 'bmi', 'age', 'sex', 'upper_ap', '

In [3]:
import json

def peek_text(path, n=3):
    try:
        with open(path, 'r', errors='replace') as f:
            return [next(f).rstrip() for _ in range(n)]
    except Exception as e:
        return [f'(not text: {e})']

ecg0 = sorted((MCD_DIR / 'ecg').glob('*.json'))[0]
with open(ecg0) as f:
    ecg = json.load(f)
print('ECG', ecg0.name, '-> keys:', list(ecg.keys()))
for k, v in ecg.items():
    if isinstance(v, list):
        head = v[0] if v else None
        if isinstance(head, dict):
            print(f'   {k}: list of dict, first keys={list(head.keys())}')
        else:
            print(f'   {k}: list(len={len(v)}) first={head}')
    elif isinstance(v, dict):
        print(f'   {k}: dict(keys={list(v.keys())})')
    else:
        print(f'   {k}: {v}')

for sub in ['ppg', 'ppg_sync', 'meta']:
    files = sorted((MCD_DIR / sub).iterdir())
    if files:
        p = files[0]
        print(f'\n{sub}/{p.name}  ({p.stat().st_size} bytes)')
        for line in peek_text(p, 3):
            print('   ', line[:120])

ECG 1020_after.json -> keys: ['@odata.context', 'frequency', 'dataX', 'data', 'segmentsData']
   @odata.context: http://192.168.6.15/api/v4/$metadata#Medmon.EcgDataApi
   frequency: 500
   dataX: list(len=15000) first=0
   data: list of dict, first keys=['title', 'values']
   segmentsData: dict(keys=['pqAverageValue', 'qtAverageValue', 'stAverageValue', 'pqIntervals', 'qtIntervals', 'stSegments'])

ppg/1020_after.PW  (607170 bytes)
    104   2023-11-13 14:18:09.873186
    105   2023-11-13 14:18:09.874189
    106   2023-11-13 14:18:09.889143

ppg_sync/1020_FullHDwebcam_after.txt  (64815 bytes)
    104 0.016954
    106 0.000997
    105 0.000998

meta/1020_FullHDwebcam_after.txt  (182459 bytes)
    
    1  2023-11-13 14:18:09.856232
    2  2023-11-13 14:18:09.888146
